In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, average_precision_score

import pandas as pd
import numpy as np

In [10]:
class AUCContrastiveLoss(nn.Module):
    def __init__(self):
        super(AUCContrastiveLoss, self).__init__()

    def forward(self, y_pred, y_true):
        pos_preds = y_pred[y_true == 1]  # 양성 샘플 예측값
        neg_preds = y_pred[y_true == 0]  # 음성 샘플 예측값
        pairwise_diff = pos_preds[:, None] - neg_preds[None, :]  # 양성-음성 조합 만들기
        loss = -torch.mean(F.logsigmoid(pairwise_diff))  # AUC 증가를 위한 Contrastive Loss
        return loss

In [24]:
train_df = pd.read_csv('simibe/train_cleaned.csv')
test_df = pd.read_csv('simibe/test_cleaned.csv')
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

categorical_cols = X.select_dtypes(include=['object']).columns
encoder_dict = {}
for col in categorical_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[col] = encoder.fit_transform(X[[col]])
    test_df[col] = encoder.transform(test_df[[col]])
    encoder_dict[col] = encoder

# 데이터 분할 (학습/검증)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X = scaler.fit_transform(X)
test_df = scaler.transform(test_df)

# PyTorch Tensor 변환 (GPU 사용 가능)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).to(device)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(test_df, dtype=torch.float32).to(device)

# 데이터 로더 생성
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

class SelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(SelfAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim)
        self.key = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        x = x.unsqueeze(1)
        Q = self.query(x)  
        K = self.key(x)    
        V = self.value(x)  
        attention_scores = self.softmax(torch.bmm(Q, K.transpose(1, 2)) / (x.shape[-1] ** 0.5))  
        return torch.bmm(attention_scores, V).squeeze(1) 
    
class FeatureAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        attn_weights = self.attention(x)
        return x * attn_weights  # Feature-wise 가중치 적용
# PyTorch MLP 모델 정의
class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super(MLPModel, self).__init__()
        self.feature_attention = FeatureAttention(input_dim)
        self.self_attention = SelfAttention(input_dim)

        self.fc1 = nn.Linear(input_dim, 1024)  # MLP 확장
        self.bn1 = nn.BatchNorm1d(1024)
        self.dropout1 = nn.Dropout(0.2)

        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(0.2)

        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.dropout3 = nn.Dropout(0.2)

        self.fc4 = nn.Linear(256, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.feature_attention(x)  # Feature Attention 적용
        x = self.self_attention(x)  # Self-Attention 적용

        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        x = self.sigmoid(self.fc4(x))
        return x

# 모델 초기화
model = MLPModel(X_train.shape[1]).to(device)
criterion = AUCContrastiveLoss() # AUC 향상용
optimizer = optim.AdamW(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
best_valid_loss = float('inf')
patience = 8
counter = 0
# 모델 학습
epochs = 100
for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    # 검증 데이터 평가
    model.eval()
    total_valid_loss = 0
    with torch.no_grad():
        y_val_proba = model(X_val_tensor).cpu().numpy().squeeze()
        y_val_pred = (y_val_proba > 0.5).astype(int)
        valid_loss = criterion(torch.tensor(y_val_proba).to(device), y_val_tensor)  # 검증 손실 계산
        total_valid_loss += valid_loss.item()
    
    auc_pr = average_precision_score(y_val, y_val_proba)
    roc_auc = roc_auc_score(y_val, y_val_proba)

    print(f"Epoch {epoch+1}, Train Loss: {total_train_loss / len(train_loader):.4f}, "
          f"Valid Loss: {total_valid_loss:.4f}, AUC PR: {auc_pr:.4f}, ROC AUC: {roc_auc:.4f}")
    
    # Early Stopping 체크
    if total_valid_loss < best_valid_loss:
        best_valid_loss = total_valid_loss
        counter = 0  # 개선되면 카운터 초기화
    else:
        counter += 1
        if counter >= patience:
            print("Early Stopping triggered!")
            break  # 조기 종료

print("Final Model Performance:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))


Epoch 1, Train Loss: 0.5833, Valid Loss: 0.5800, AUC PR: 0.4263, ROC AUC: 0.7265
Epoch 2, Train Loss: 0.5776, Valid Loss: 0.5798, AUC PR: 0.4207, ROC AUC: 0.7238
Epoch 3, Train Loss: 0.5763, Valid Loss: 0.5781, AUC PR: 0.4271, ROC AUC: 0.7279
Epoch 4, Train Loss: 0.5760, Valid Loss: 0.5809, AUC PR: 0.4172, ROC AUC: 0.7240
Epoch 5, Train Loss: 0.5749, Valid Loss: 0.5781, AUC PR: 0.4282, ROC AUC: 0.7292
Epoch 6, Train Loss: 0.5748, Valid Loss: 0.5799, AUC PR: 0.4244, ROC AUC: 0.7267
Epoch 7, Train Loss: 0.5735, Valid Loss: 0.5786, AUC PR: 0.4277, ROC AUC: 0.7296
Epoch 8, Train Loss: 0.5737, Valid Loss: 0.5791, AUC PR: 0.4239, ROC AUC: 0.7280
Epoch 9, Train Loss: 0.5735, Valid Loss: 0.5784, AUC PR: 0.4254, ROC AUC: 0.7291
Epoch 10, Train Loss: 0.5726, Valid Loss: 0.5804, AUC PR: 0.4219, ROC AUC: 0.7259
Epoch 11, Train Loss: 0.5722, Valid Loss: 0.5778, AUC PR: 0.4229, ROC AUC: 0.7285
Epoch 12, Train Loss: 0.5725, Valid Loss: 0.5796, AUC PR: 0.4220, ROC AUC: 0.7264
Epoch 13, Train Loss: 0.5

In [ ]:
# 전체 데이터로 재학습
model_full = MLPModel(X.shape[1]).to(device)
model_full.load_state_dict(model.state_dict())
model_full.eval()
with torch.no_grad():
    y_pred_proba = model_full(X_test_tensor).cpu().numpy().squeeze()

# 제출 파일 생성
sample_submission = pd.read_csv('/mnt/data/sample_submission.csv')
sample_submission['probability'] = y_pred_proba
sample_submission.to_csv('./mlp_pytorch_submission.csv', index=False)
